<a href="https://colab.research.google.com/github/seenu-2006/entity-matching-ml-pipeline/blob/main/ML_Based_Entity_Resolution_%26_Matching_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ================================================================
# CELL 1 - DATASET CHECK
# ================================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import pandas as pd

BASE = "/content/drive/MyDrive/student_resource"

FILES = {
    "train_source1": f"{BASE}/dataset/train/train_source1.tsv",
    "train_source2": f"{BASE}/dataset/train/train_source2.tsv",
    "train_source3": f"{BASE}/dataset/train/train_source3.tsv",
    "ground_truth":  f"{BASE}/dataset/train/train_ground_truth.tsv",
    "test_source1":  f"{BASE}/dataset/test/test_source1.tsv",
    "test_source2":  f"{BASE}/dataset/test/test_source2.tsv",
    "test_source3":  f"{BASE}/dataset/test/test_source3.tsv",
    "validator":     f"{BASE}/utils/validate_submission.py"
}

print("=" * 65)
print("DATASET CHECK")
print("=" * 65)

if not os.path.exists(BASE):
    raise FileNotFoundError(f"Base path not found:\n{BASE}")

for name, path in FILES.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"{name} not found:\n{path}")
    size = os.path.getsize(path) / (1024 * 1024)
    print(f"✅ {name:15} {size:8.2f} MB")

# Check headers only — don't load huge files into RAM
print("\nChecking columns...")

for name in [
    "train_source1",
    "train_source2",
    "train_source3",
    "ground_truth",
    "test_source1",
    "test_source2",
    "test_source3"
]:
    cols = pd.read_csv(
        FILES[name],
        sep="\t",
        nrows=3
    ).columns.tolist()

    print(f"{name:15}: {cols}")

print("\n" + "=" * 65)
print("CELL 1 COMPLETE ✅")
print("BASE:", BASE)
print("Original entity IDs will NOT be changed 🔒")
print("=" * 65)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DATASET CHECK
✅ train_source1     200.34 MB
✅ train_source2     466.63 MB
✅ train_source3     480.37 MB
✅ ground_truth      121.13 MB
✅ test_source1      166.91 MB
✅ test_source2      485.86 MB
✅ test_source3      482.56 MB
✅ validator           0.01 MB

Checking columns...
train_source1  : ['entity_id', 'business_name', 'business_address', 'country']
train_source2  : ['entity_id', 'business_name', 'business_address', 'country']
train_source3  : ['entity_id', 'business_name', 'business_address', 'country']
ground_truth   : ['source1_entity_id', 'matched_entity_ids']
test_source1   : ['entity_id', 'business_name', 'business_address', 'country']
test_source2   : ['entity_id', 'business_name', 'business_address', 'country']
test_source3   : ['entity_id', 'business_name', 'business_address', 'country']

CELL 1 COMPLETE ✅
BASE: /content/drive/MyDrive/student_resou

In [2]:
# ================================================================
# CELL 2 - FREQUENCY FILTERED + RAM SAFE BLOCKING
# ================================================================

import os
import re
import gc
import csv
import sqlite3
import time
import pandas as pd

# ================================================================
# PATHS
# ================================================================

BASE = "/content/drive/MyDrive/student_resource"

TRAIN1 = f"{BASE}/dataset/train/train_source1.tsv"
TRAIN2 = f"{BASE}/dataset/train/train_source2.tsv"
TRAIN3 = f"{BASE}/dataset/train/train_source3.tsv"

BLOCK_DIR = f"{BASE}/blocking"
os.makedirs(BLOCK_DIR, exist_ok=True)

DB_FILE = "/content/frequency_filtered_blocking.db"
CANDIDATE_FILE = f"{BLOCK_DIR}/training_candidates_filtered.tsv"

# ================================================================
# SETTINGS
# ================================================================

SOURCE_CHUNK = 100_000
SOURCE1_CHUNK = 2_000
FETCH_SIZE = 5_000

# IMPORTANT:
# A blocking key appearing more than this many times
# is considered too common and is ignored.
MAX_KEY_FREQUENCY = 100


# ================================================================
# CHECK FILES
# ================================================================

print("=" * 70)
print("CELL 2 - FREQUENCY FILTERED BLOCKING")
print("=" * 70)

for path in [TRAIN1, TRAIN2, TRAIN3]:

    if not os.path.exists(path):
        raise FileNotFoundError(path)

print("All training files found ✅")


# ================================================================
# NORMALIZATION
# ================================================================

def normalize_text(x):

    if pd.isna(x):
        return ""

    x = str(x).lower()

    x = re.sub(
        r"[^a-z0-9]+",
        " ",
        x
    )

    return " ".join(x.split())


def make_keys(name, country):

    name = normalize_text(name)
    country = normalize_text(country)

    if not name:
        return []

    compact = name.replace(" ", "")

    keys = []

    # Strong exact-name key
    keys.append(
        "EXACT|" + name
    )

    # Country + 5-character prefix
    if country and len(compact) >= 5:

        keys.append(
            "CP5|" +
            country +
            "|" +
            compact[:5]
        )

    return keys


# ================================================================
# FRESH DATABASE
# ================================================================

if os.path.exists(DB_FILE):

    os.remove(DB_FILE)

print("\nCreating fresh SQLite database...")

conn = sqlite3.connect(DB_FILE)

conn.execute(
    "PRAGMA journal_mode=OFF"
)

conn.execute(
    "PRAGMA synchronous=OFF"
)

conn.execute(
    "PRAGMA temp_store=FILE"
)

conn.execute(
    "PRAGMA cache_size=-15000"
)

# Raw blocking keys first
conn.execute("""
CREATE TABLE raw_blocking (
    blocking_key TEXT NOT NULL,
    target_id TEXT NOT NULL,
    target_source TEXT NOT NULL
)
""")

conn.commit()

print("SQLite database ready ✅")


# ================================================================
# INDEX SOURCE
# ================================================================

def index_source(path, source_name):

    print("\n" + "=" * 70)
    print(f"INDEXING {source_name.upper()}")
    print("=" * 70)

    total = 0
    key_count = 0

    start = time.time()

    cursor = conn.cursor()

    for chunk in pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        usecols=[
            "entity_id",
            "business_name",
            "country"
        ],
        chunksize=SOURCE_CHUNK
    ):

        rows = []

        for entity_id, name, country in zip(
            chunk["entity_id"],
            chunk["business_name"],
            chunk["country"]
        ):

            # ORIGINAL ENTITY ID PRESERVED
            for key in make_keys(
                name,
                country
            ):

                rows.append(
                    (
                        key,
                        entity_id,
                        source_name
                    )
                )

        if rows:

            cursor.executemany(
                """
                INSERT INTO raw_blocking
                (
                    blocking_key,
                    target_id,
                    target_source
                )
                VALUES (?, ?, ?)
                """,
                rows
            )

            conn.commit()

            key_count += len(rows)

        total += len(chunk)

        if total % 500_000 == 0:

            elapsed = (
                time.time() - start
            ) / 60

            print(
                f"{source_name}: "
                f"{total:,} records | "
                f"{key_count:,} keys | "
                f"{elapsed:.2f} min"
            )

        del rows
        del chunk

        gc.collect()

    print(
        f"\n{source_name} COMPLETE ✅"
    )

    print(
        f"Records : {total:,}"
    )

    print(
        f"Keys    : {key_count:,}"
    )

    return total


# ================================================================
# INDEX SOURCE2 + SOURCE3
# ================================================================

source2_count = index_source(
    TRAIN2,
    "source2"
)

source3_count = index_source(
    TRAIN3,
    "source3"
)


# ================================================================
# FREQUENCY ANALYSIS
# ================================================================

print("\n" + "=" * 70)
print("FILTERING COMMON BLOCKING KEYS")
print("=" * 70)

start = time.time()

# Keep only informative keys.
#
# Example:
# key appears 3 times  -> KEEP
# key appears 80 times -> KEEP
# key appears 5,000 times -> REMOVE

conn.execute("""
CREATE TABLE blocking_index AS

SELECT
    r.blocking_key,
    r.target_id,
    r.target_source

FROM raw_blocking r

INNER JOIN
(
    SELECT
        blocking_key

    FROM raw_blocking

    GROUP BY blocking_key

    HAVING COUNT(*) <= ?
) good

ON r.blocking_key = good.blocking_key
""", (MAX_KEY_FREQUENCY,))

conn.commit()

print(
    f"Keys with frequency <= "
    f"{MAX_KEY_FREQUENCY} retained ✅"
)

# Raw table no longer needed
conn.execute(
    "DROP TABLE raw_blocking"
)

conn.commit()

print("Raw blocking table removed ✅")


# ================================================================
# INDEX FILTERED KEYS
# ================================================================

print("\nCreating filtered blocking index...")

conn.execute("""
CREATE INDEX idx_blocking_key
ON blocking_index(blocking_key)
""")

conn.commit()

print("Filtered SQLite index created ✅")


# ================================================================
# SHOW BLOCKING STATISTICS
# ================================================================

filtered_keys = conn.execute(
    "SELECT COUNT(*) FROM blocking_index"
).fetchone()[0]

unique_keys = conn.execute(
    """
    SELECT COUNT(DISTINCT blocking_key)
    FROM blocking_index
    """
).fetchone()[0]

print("\nFiltered blocking statistics:")
print(
    f"Rows : {filtered_keys:,}"
)
print(
    f"Unique keys : {unique_keys:,}"
)


# ================================================================
# SOURCE-1 TEMP TABLE
# ================================================================

conn.execute("""
CREATE TEMP TABLE source1_keys (
    source1_id TEXT NOT NULL,
    blocking_key TEXT NOT NULL
)
""")

conn.execute("""
CREATE INDEX idx_source1_key
ON source1_keys(blocking_key)
""")

conn.commit()


# ================================================================
# OUTPUT FILE
# ================================================================

if os.path.exists(CANDIDATE_FILE):

    os.remove(CANDIDATE_FILE)

with open(
    CANDIDATE_FILE,
    "w",
    encoding="utf-8",
    newline=""
) as f:

    writer = csv.writer(
        f,
        delimiter="\t"
    )

    writer.writerow([
        "source1_id",
        "target_id",
        "target_source"
    ])


# ================================================================
# CANDIDATE GENERATION
# ================================================================

print("\n" + "=" * 70)
print("GENERATING FILTERED CANDIDATES")
print("=" * 70)

start = time.time()

source1_processed = 0
candidate_count = 0

for chunk_no, chunk in enumerate(
    pd.read_csv(
        TRAIN1,
        sep="\t",
        dtype=str,
        usecols=[
            "entity_id",
            "business_name",
            "country"
        ],
        chunksize=SOURCE1_CHUNK
    ),
    start=1
):

    # Clear previous batch
    conn.execute(
        "DELETE FROM source1_keys"
    )

    key_rows = []

    for source1_id, name, country in zip(
        chunk["entity_id"],
        chunk["business_name"],
        chunk["country"]
    ):

        for key in make_keys(
            name,
            country
        ):

            key_rows.append(
                (
                    source1_id,
                    key
                )
            )

    if key_rows:

        conn.executemany(
            """
            INSERT INTO source1_keys
            (
                source1_id,
                blocking_key
            )
            VALUES (?, ?)
            """,
            key_rows
        )

        conn.commit()

    # ------------------------------------------------------------
    # FILTERED JOIN
    # ------------------------------------------------------------

    sql = """
    SELECT
        s.source1_id,
        b.target_id,
        b.target_source

    FROM source1_keys s

    INNER JOIN blocking_index b
        ON s.blocking_key = b.blocking_key

    GROUP BY
        s.source1_id,
        b.target_id,
        b.target_source
    """

    cursor = conn.execute(sql)

    # ------------------------------------------------------------
    # STREAM TO DISK
    # ------------------------------------------------------------

    while True:

        rows = cursor.fetchmany(
            FETCH_SIZE
        )

        if not rows:
            break

        with open(
            CANDIDATE_FILE,
            "a",
            encoding="utf-8",
            newline=""
        ) as f:

            writer = csv.writer(
                f,
                delimiter="\t"
            )

            writer.writerows(rows)

        candidate_count += len(rows)

        del rows

        gc.collect()

    source1_processed += len(chunk)

    elapsed = (
        time.time() - start
    ) / 60

    print(
        f"Chunk {chunk_no:03d} | "
        f"Source-1: {source1_processed:,} | "
        f"Candidates: {candidate_count:,} | "
        f"Time: {elapsed:.2f} min"
    )

    del key_rows
    del chunk

    gc.collect()


# ================================================================
# FINAL
# ================================================================

conn.close()

gc.collect()

candidate_size = (
    os.path.getsize(CANDIDATE_FILE)
    / (1024 * 1024)
)

db_size = (
    os.path.getsize(DB_FILE)
    / (1024 * 1024)
)

print("\n" + "=" * 70)
print("CELL 2 RESULTS")
print("=" * 70)

print(
    f"Source-1 records : {source1_processed:,}"
)

print(
    f"Source-2 records : {source2_count:,}"
)

print(
    f"Source-3 records : {source3_count:,}"
)

print(
    f"Filtered keys    : {unique_keys:,}"
)

print(
    f"Candidate rows   : {candidate_count:,}"
)

print(
    f"Candidate file   : {candidate_size:.2f} MB"
)

print(
    f"SQLite DB        : {db_size:.2f} MB"
)

print(
    f"\nOutput:"
)

print(
    CANDIDATE_FILE
)

print("\nOriginal entity IDs preserved 🔒")
print("Source-2 complete ✅")
print("Source-3 complete ✅")
print("Common-key filtering enabled ✅")
print("RAM-safe candidate streaming enabled ✅")

print("\n" + "=" * 70)
print("CELL 2 COMPLETE ✅")
print("=" * 70)

CELL 2 - FREQUENCY FILTERED BLOCKING
All training files found ✅

Creating fresh SQLite database...
SQLite database ready ✅

INDEXING SOURCE2
source2: 500,000 records | 908,830 keys | 0.14 min
source2: 1,000,000 records | 1,817,736 keys | 0.25 min
source2: 1,500,000 records | 2,725,743 keys | 0.40 min
source2: 2,000,000 records | 3,633,190 keys | 0.52 min
source2: 2,500,000 records | 4,540,515 keys | 0.67 min
source2: 3,000,000 records | 5,448,558 keys | 0.78 min
source2: 3,500,000 records | 6,356,815 keys | 0.92 min
source2: 4,000,000 records | 7,264,281 keys | 1.03 min
source2: 4,500,000 records | 8,172,269 keys | 1.19 min
source2: 5,000,000 records | 9,080,491 keys | 1.30 min

source2 COMPLETE ✅
Records : 5,034,616
Keys    : 9,143,330

INDEXING SOURCE3
source3: 500,000 records | 950,543 keys | 0.14 min
source3: 1,000,000 records | 1,901,305 keys | 0.26 min
source3: 1,500,000 records | 2,851,717 keys | 0.40 min
source3: 2,000,000 records | 3,801,855 keys | 0.51 min
source3: 2,500,000 

In [4]:
# ================================================================
# CELL 3 - FINAL RAM-SAFE ML FEATURE GENERATION
# ================================================================

!pip -q install rapidfuzz

import os
import gc
import csv
import sqlite3
import time
import numpy as np
import pandas as pd

from rapidfuzz.fuzz import ratio


# ================================================================
# PATHS
# ================================================================

BASE = "/content/drive/MyDrive/student_resource"

SOURCE1 = f"{BASE}/dataset/train/train_source1.tsv"
SOURCE2 = f"{BASE}/dataset/train/train_source2.tsv"
SOURCE3 = f"{BASE}/dataset/train/train_source3.tsv"

CANDIDATES = (
    f"{BASE}/blocking/"
    f"training_candidates_filtered.tsv"
)

FEATURE_FILE = (
    f"{BASE}/blocking/"
    f"ml_features_filtered.tsv"
)

DB_FILE = "/content/feature_lookup.db"


# ================================================================
# SETTINGS
# ================================================================

CANDIDATE_CHUNK = 25_000
SOURCE_CHUNK = 100_000


print("=" * 70)
print("CELL 3 - FINAL RAM-SAFE ML FEATURE GENERATION")
print("=" * 70)


# ================================================================
# CHECK FILES
# ================================================================

for path in [
    SOURCE1,
    SOURCE2,
    SOURCE3,
    CANDIDATES
]:

    if not os.path.exists(path):

        raise FileNotFoundError(
            f"Missing file: {path}"
        )

print("All required files found ✅")


# ================================================================
# CREATE LOCAL LOOKUP DATABASE
# ================================================================

if os.path.exists(DB_FILE):

    os.remove(DB_FILE)


print("\nCreating local feature lookup database...")

db = sqlite3.connect(DB_FILE)

db.execute("""
CREATE TABLE entities (

    entity_id TEXT PRIMARY KEY,

    business_name TEXT,

    business_address TEXT,

    country TEXT,

    source TEXT
)
""")

db.commit()


# ================================================================
# LOAD SOURCE INTO SQLITE
# ================================================================

def index_source(
    path,
    source_name
):

    print(
        f"\nIndexing {source_name}..."
    )

    total = 0

    for chunk in pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        usecols=[
            "entity_id",
            "business_name",
            "business_address",
            "country"
        ],
        chunksize=SOURCE_CHUNK
    ):

        chunk = chunk.fillna("")

        rows = []

        for row in chunk.itertuples(
            index=False,
            name=None
        ):

            rows.append(
                (
                    row[0],
                    row[1],
                    row[2],
                    row[3],
                    source_name
                )
            )

        db.executemany(
            """
            INSERT OR REPLACE INTO entities
            VALUES (?, ?, ?, ?, ?)
            """,
            rows
        )

        db.commit()

        total += len(chunk)

        if total % 500_000 == 0:

            print(
                f"{source_name}: "
                f"{total:,} records"
            )

        del rows
        del chunk

        gc.collect()

    print(
        f"{source_name} complete: "
        f"{total:,}"
    )


# ================================================================
# INDEX ALL SOURCES
# ================================================================

index_source(
    SOURCE1,
    "source1"
)

index_source(
    SOURCE2,
    "source2"
)

index_source(
    SOURCE3,
    "source3"
)


# ================================================================
# DATABASE INDEX
# ================================================================

print("\nCreating database index...")

db.execute("""
CREATE INDEX idx_entity_id
ON entities(entity_id)
""")

db.commit()

print("Database lookup ready ✅")


# ================================================================
# OUTPUT FILE
# ================================================================

if os.path.exists(
    FEATURE_FILE
):

    os.remove(
        FEATURE_FILE
    )


feature_columns = [

    "source1_id",
    "target_id",
    "target_source",

    "name_similarity",
    "address_similarity",

    "country_match",
    "exact_name",

    "name_length_diff",
    "address_length_diff"
]


# ================================================================
# WRITE HEADER
# ================================================================

with open(
    FEATURE_FILE,
    "w",
    encoding="utf-8",
    newline=""
) as f:

    writer = csv.writer(
        f,
        delimiter="\t"
    )

    writer.writerow(
        feature_columns
    )


# ================================================================
# FEATURE GENERATION
# ================================================================

print("\n" + "=" * 70)
print("GENERATING ML FEATURES")
print("=" * 70)

start = time.time()

processed = 0


for chunk_no, candidates in enumerate(

    pd.read_csv(
        CANDIDATES,
        sep="\t",
        dtype=str,
        chunksize=CANDIDATE_CHUNK
    ),

    start=1
):

    candidates = candidates.fillna("")


    # ------------------------------------------------------------
    # GET UNIQUE SOURCE-1 IDs
    # ------------------------------------------------------------

    source1_ids = (
        candidates["source1_id"]
        .drop_duplicates()
        .tolist()
    )


    # ------------------------------------------------------------
    # GET UNIQUE TARGET IDs
    # ------------------------------------------------------------

    target_ids = (
        candidates["target_id"]
        .drop_duplicates()
        .tolist()
    )


    # ------------------------------------------------------------
    # DATABASE LOOKUP
    # ------------------------------------------------------------

    all_ids = (
        source1_ids +
        target_ids
    )

    placeholders = ",".join(
        ["?"] * len(all_ids)
    )

    query = f"""
    SELECT
        entity_id,
        business_name,
        business_address,
        country,
        source
    FROM entities
    WHERE entity_id IN ({placeholders})
    """

    lookup = pd.read_sql_query(
        query,
        db,
        params=all_ids
    )

    lookup = lookup.set_index(
        "entity_id"
    )


    # ------------------------------------------------------------
    # FEATURE CALCULATION
    # ------------------------------------------------------------

    output_rows = []


    for row in candidates.itertuples(
        index=False
    ):

        s1_id = row.source1_id
        target_id = row.target_id
        target_source = row.target_source


        # --------------------------------------------
        # SOURCE-1 DATA
        # --------------------------------------------

        if s1_id not in lookup.index:

            continue

        s1 = lookup.loc[s1_id]


        # --------------------------------------------
        # TARGET DATA
        # --------------------------------------------

        if target_id not in lookup.index:

            continue

        target = lookup.loc[target_id]


        # --------------------------------------------
        # TEXT
        # --------------------------------------------

        s1_name = str(
            s1["business_name"]
        )

        target_name = str(
            target["business_name"]
        )

        s1_address = str(
            s1["business_address"]
        )

        target_address = str(
            target["business_address"]
        )

        s1_country = str(
            s1["country"]
        ).lower().strip()

        target_country = str(
            target["country"]
        ).lower().strip()


        # --------------------------------------------
        # NAME SIMILARITY
        # --------------------------------------------

        name_sim = (
            ratio(
                s1_name,
                target_name
            ) / 100.0
        )


        # --------------------------------------------
        # ADDRESS SIMILARITY
        # --------------------------------------------

        address_sim = (
            ratio(
                s1_address,
                target_address
            ) / 100.0
        )


        # --------------------------------------------
        # COUNTRY
        # --------------------------------------------

        country_match = int(
            s1_country ==
            target_country
        )


        # --------------------------------------------
        # EXACT NAME
        # --------------------------------------------

        exact_name = int(
            s1_name.lower().strip()
            ==
            target_name.lower().strip()
        )


        # --------------------------------------------
        # LENGTH DIFFERENCES
        # --------------------------------------------

        name_length_diff = abs(
            len(s1_name)
            -
            len(target_name)
        )

        address_length_diff = abs(
            len(s1_address)
            -
            len(target_address)
        )


        output_rows.append([

            s1_id,

            target_id,

            target_source,

            name_sim,

            address_sim,

            country_match,

            exact_name,

            name_length_diff,

            address_length_diff

        ])


    # ------------------------------------------------------------
    # WRITE CHUNK
    # ------------------------------------------------------------

    if output_rows:

        with open(
            FEATURE_FILE,
            "a",
            encoding="utf-8",
            newline=""
        ) as f:

            writer = csv.writer(
                f,
                delimiter="\t"
            )

            writer.writerows(
                output_rows
            )


    processed += len(candidates)

    elapsed = (
        time.time() - start
    ) / 60


    print(
        f"Chunk {chunk_no:04d} | "
        f"Processed: {processed:,} | "
        f"Time: {elapsed:.2f} min"
    )


    # ------------------------------------------------------------
    # MEMORY CLEANUP
    # ------------------------------------------------------------

    del candidates
    del source1_ids
    del target_ids
    del all_ids
    del lookup
    del output_rows

    gc.collect()


# ================================================================
# CLOSE DATABASE
# ================================================================

db.close()

gc.collect()


# ================================================================
# RESULT
# ================================================================

feature_size = (
    os.path.getsize(
        FEATURE_FILE
    )
    /
    (1024 * 1024)
)


print("\n" + "=" * 70)
print("CELL 3 RESULTS")
print("=" * 70)

print(
    f"Processed candidates : "
    f"{processed:,}"
)

print(
    f"Feature file         : "
    f"{FEATURE_FILE}"
)

print(
    f"Feature file size    : "
    f"{feature_size:.2f} MB"
)

print(
    "\nOriginal Source-1 IDs preserved 🔒"
)

print(
    "Original target IDs preserved 🔒"
)

print(
    "Chunk-wise processing ✅"
)

print(
    "RAM-safe processing ✅"
)

print("=" * 70)
print("CELL 3 COMPLETE ✅")
print("=" * 70)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 31.8 MB/s eta 0:00:00
CELL 3 - FINAL RAM-SAFE ML FEATURE GENERATION
All required files found ✅

Creating local feature lookup database...

Indexing source1...
source1: 500,000 records
source1: 1,000,000 records
source1: 1,500,000 records
source1: 2,000,000 records
source1 complete: 2,206,821

Indexing source2...
source2: 500,000 records
source2: 1,000,000 records
source2: 1,500,000 records
source2: 2,000,000 records
source2: 2,500,000 records
source2: 3,000,000 records
source2: 3,500,000 records
source2: 4,000,000 records
source2: 4,500,000 records
source2: 5,000,000 records
source2 complete: 5,034,616

Indexing source3...
source3: 500,000 records
source3: 1,000,000 records
source3: 1,500,000 records
source3: 2,000,000 records
source3: 2,500,000 records
source3: 3,000,000 records
source3: 3,500,000 records
source3: 4,000,000 records
source3: 4,500,000 records
source3: 5,000,000 records
source3 complete: 5,285,603

Creating database

In [6]:
# ======================================================================
# CELL 4 - RAM-SAFE ML TRAINING WITH PSEUDO LABELS
# ======================================================================

import os
import gc
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

print("=" * 70)
print("CELL 4 - RAM-SAFE ML TRAINING")
print("=" * 70)

FEATURE_FILE = "/content/drive/MyDrive/student_resource/blocking/ml_features_filtered.tsv"

if not os.path.exists(FEATURE_FILE):
    raise FileNotFoundError(f"Feature file not found: {FEATURE_FILE}")

print("Feature file found ✅")
print(f"Feature file size : {os.path.getsize(FEATURE_FILE) / (1024**3):.2f} GB")

# ----------------------------------------------------------------------
# STEP 1 - Read only header
# ----------------------------------------------------------------------

header = pd.read_csv(
    FEATURE_FILE,
    sep="\t",
    nrows=0
)

print("\nAvailable columns:")
print(list(header.columns))

required_features = [
    "name_similarity",
    "address_similarity",
    "country_match",
    "exact_name",
    "name_length_diff",
    "address_length_diff"
]

missing = [c for c in required_features if c not in header.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("\nRequired features found ✅")


# ----------------------------------------------------------------------
# STEP 2 - Create pseudo-labels
# ----------------------------------------------------------------------
#
# Since there is NO ground-truth target column,
# create labels using similarity rules.
#
# Positive = strong entity match
# Negative = weak entity match
# Ambiguous records are ignored.
# ----------------------------------------------------------------------

print("\nCreating pseudo-labels...")

CHUNK_SIZE = 100_000

X_parts = []
y_parts = []

total_rows = 0
positive_count = 0
negative_count = 0
ignored_count = 0

for chunk in pd.read_csv(
    FEATURE_FILE,
    sep="\t",
    chunksize=CHUNK_SIZE
):

    total_rows += len(chunk)

    # --------------------------------------------------------------
    # Convert numeric columns safely
    # --------------------------------------------------------------

    for col in required_features:
        chunk[col] = pd.to_numeric(
            chunk[col],
            errors="coerce"
        )

    chunk = chunk.dropna(subset=required_features)

    # --------------------------------------------------------------
    # Normalize length differences
    # --------------------------------------------------------------

    name_len = chunk["name_length_diff"].abs()
    address_len = chunk["address_length_diff"].abs()

    # --------------------------------------------------------------
    # Strong positive matches
    # --------------------------------------------------------------

    positive = (
        (
            (chunk["name_similarity"] >= 0.90) &
            (chunk["address_similarity"] >= 0.70)
        )
        |
        (
            (chunk["exact_name"] == 1) &
            (chunk["country_match"] == 1)
        )
        |
        (
            (chunk["name_similarity"] >= 0.95) &
            (chunk["country_match"] == 1)
        )
    )

    # --------------------------------------------------------------
    # Strong negative matches
    # --------------------------------------------------------------

    negative = (
        (
            (chunk["name_similarity"] < 0.40) &
            (chunk["address_similarity"] < 0.30)
        )
        |
        (
            (chunk["name_similarity"] < 0.50) &
            (chunk["country_match"] == 0)
        )
    )

    # --------------------------------------------------------------
    # Keep only confident examples
    # --------------------------------------------------------------

    selected = positive | negative

    if selected.any():

        selected_chunk = chunk.loc[
            selected,
            required_features
        ].copy()

        labels = np.where(
            positive[selected],
            1,
            0
        )

        X_parts.append(selected_chunk)
        y_parts.append(labels)

        positive_count += int((labels == 1).sum())
        negative_count += int((labels == 0).sum())

    ignored_count += int((~selected).sum())

    if total_rows % 1_000_000 < CHUNK_SIZE:
        print(
            f"Processed : {total_rows:,} | "
            f"Positive : {positive_count:,} | "
            f"Negative : {negative_count:,}"
        )

    del chunk
    gc.collect()


print("\n" + "=" * 70)
print("PSEUDO-LABEL GENERATION COMPLETE")
print("=" * 70)

print(f"Total rows processed : {total_rows:,}")
print(f"Positive samples     : {positive_count:,}")
print(f"Negative samples     : {negative_count:,}")
print(f"Ignored ambiguous    : {ignored_count:,}")


# ----------------------------------------------------------------------
# STEP 3 - Combine selected training data
# ----------------------------------------------------------------------

if positive_count == 0 or negative_count == 0:
    raise ValueError(
        "Could not create both positive and negative classes. "
        "Pseudo-label thresholds need adjustment."
    )

X = pd.concat(
    X_parts,
    ignore_index=True
)

y = np.concatenate(y_parts)

# Free temporary lists
del X_parts
del y_parts
gc.collect()

print("\nTraining dataset created ✅")
print(f"Training samples : {len(X):,}")
print(f"Features         : {X.shape[1]}")

print("\nClass distribution:")
print(pd.Series(y).value_counts())


# ----------------------------------------------------------------------
# STEP 4 - Train/Test split
# ----------------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTrain/Test split complete ✅")
print(f"Train samples : {len(X_train):,}")
print(f"Test samples  : {len(X_test):,}")


# ----------------------------------------------------------------------
# STEP 5 - RAM-safe Random Forest
# ----------------------------------------------------------------------

print("\nTraining Random Forest...")

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=2,
    class_weight="balanced_subsample"
)

model.fit(
    X_train,
    y_train
)

print("Model training complete ✅")


# ----------------------------------------------------------------------
# STEP 6 - Evaluation
# ----------------------------------------------------------------------

print("\nEvaluating model...")

y_pred = model.predict(X_test)

print("\nClassification Report")
print("=" * 70)

print(
    classification_report(
        y_test,
        y_pred,
        target_names=[
            "Non-Match",
            "Match"
        ]
    )
)

print("\nConfusion Matrix")
print("=" * 70)

print(
    confusion_matrix(
        y_test,
        y_pred
    )
)


# ----------------------------------------------------------------------
# STEP 7 - Feature importance
# ----------------------------------------------------------------------

importance = pd.DataFrame({
    "feature": required_features,
    "importance": model.feature_importances_
})

importance = importance.sort_values(
    "importance",
    ascending=False
)

print("\nFeature Importance")
print("=" * 70)
print(importance.to_string(index=False))


# ----------------------------------------------------------------------
# STEP 8 - Save model
# ----------------------------------------------------------------------

MODEL_FILE = "/content/drive/MyDrive/student_resource/blocking/entity_matching_model.pkl"

import joblib

joblib.dump(
    model,
    MODEL_FILE
)

print("\nModel saved ✅")
print(MODEL_FILE)

# Cleanup
del X
del X_train
del X_test
del y_train
del y_test
del y
del model

gc.collect()

print("\n" + "=" * 70)
print("CELL 4 COMPLETE ✅")
print("=" * 70)

CELL 4 - RAM-SAFE ML TRAINING
Feature file found ✅
Feature file size : 1.97 GB

Available columns:
['source1_id', 'target_id', 'target_source', 'name_similarity', 'address_similarity', 'country_match', 'exact_name', 'name_length_diff', 'address_length_diff']

Required features found ✅

Creating pseudo-labels...
Processed : 1,000,000 | Positive : 387,545 | Negative : 100,292
Processed : 2,000,000 | Positive : 778,131 | Negative : 199,880
Processed : 3,000,000 | Positive : 1,166,277 | Negative : 299,541
Processed : 4,000,000 | Positive : 1,557,636 | Negative : 398,444
Processed : 5,000,000 | Positive : 1,943,078 | Negative : 499,225
Processed : 6,000,000 | Positive : 2,334,656 | Negative : 598,364
Processed : 7,000,000 | Positive : 2,722,356 | Negative : 699,102
Processed : 8,000,000 | Positive : 3,112,199 | Negative : 799,283
Processed : 9,000,000 | Positive : 3,502,542 | Negative : 898,924
Processed : 10,000,000 | Positive : 3,893,634 | Negative : 997,969
Processed : 11,000,000 | Posit

In [7]:
# ======================================================================
# CELL 5 - RAM-SAFE FINAL PREDICTION & SUBMISSION FILE
# ======================================================================

import os
import gc
import joblib
import pandas as pd
import numpy as np

print("=" * 70)
print("CELL 5 - RAM-SAFE FINAL PREDICTION")
print("=" * 70)

# ----------------------------------------------------------------------
# PATHS
# ----------------------------------------------------------------------

FEATURE_FILE = "/content/drive/MyDrive/student_resource/blocking/ml_features_filtered.tsv"

MODEL_FILE = "/content/drive/MyDrive/student_resource/blocking/entity_matching_model.pkl"

OUTPUT_FILE = "/content/drive/MyDrive/student_resource/blocking/final_submission.csv"

# ----------------------------------------------------------------------
# CHECK FILES
# ----------------------------------------------------------------------

if not os.path.exists(FEATURE_FILE):
    raise FileNotFoundError(f"Feature file not found:\n{FEATURE_FILE}")

if not os.path.exists(MODEL_FILE):
    raise FileNotFoundError(f"Model file not found:\n{MODEL_FILE}")

print("Feature file found ✅")
print("Model file found ✅")

print(f"Feature file size : {os.path.getsize(FEATURE_FILE) / (1024**3):.2f} GB")

# ----------------------------------------------------------------------
# LOAD MODEL
# ----------------------------------------------------------------------

print("\nLoading trained model...")

model = joblib.load(MODEL_FILE)

print("Model loaded ✅")

# ----------------------------------------------------------------------
# FEATURES USED DURING TRAINING
# ----------------------------------------------------------------------

FEATURE_COLUMNS = [
    "name_similarity",
    "address_similarity",
    "country_match",
    "exact_name",
    "name_length_diff",
    "address_length_diff"
]

# ----------------------------------------------------------------------
# OUTPUT COLUMNS
# ----------------------------------------------------------------------

USE_COLUMNS = [
    "source1_id",
    "target_id",
    "target_source"
] + FEATURE_COLUMNS

# ----------------------------------------------------------------------
# CHUNK SETTINGS
# ----------------------------------------------------------------------

CHUNK_SIZE = 100_000

# High confidence threshold
# Start with 0.80
MATCH_THRESHOLD = 0.80

print(f"\nChunk size       : {CHUNK_SIZE:,}")
print(f"Match threshold  : {MATCH_THRESHOLD}")

# ----------------------------------------------------------------------
# TEMP FILE
# ----------------------------------------------------------------------

TEMP_FILE = "/content/drive/MyDrive/student_resource/blocking/final_matches_temp.tsv"

if os.path.exists(TEMP_FILE):
    os.remove(TEMP_FILE)

# ----------------------------------------------------------------------
# PROCESS CHUNKS
# ----------------------------------------------------------------------

total_processed = 0
total_matches = 0

first_write = True

print("\nStarting chunk-wise prediction...")
print("=" * 70)

reader = pd.read_csv(
    FEATURE_FILE,
    sep="\t",
    usecols=USE_COLUMNS,
    chunksize=CHUNK_SIZE,
    low_memory=True
)

for chunk_no, chunk in enumerate(reader, start=1):

    # --------------------------------------------------------------
    # Make sure numeric columns are numeric
    # --------------------------------------------------------------

    X = chunk[FEATURE_COLUMNS].apply(
        pd.to_numeric,
        errors="coerce"
    ).fillna(0)

    # --------------------------------------------------------------
    # Predict probability
    # --------------------------------------------------------------

    probabilities = model.predict_proba(X)[:, 1]

    chunk["match_probability"] = probabilities

    # --------------------------------------------------------------
    # Keep high-confidence matches
    # --------------------------------------------------------------

    matches = chunk[
        chunk["match_probability"] >= MATCH_THRESHOLD
    ].copy()

    if len(matches) > 0:

        # Keep only required information
        matches = matches[
            [
                "source1_id",
                "target_id",
                "target_source",
                "match_probability"
            ]
        ]

        # ----------------------------------------------------------
        # Append to temporary file
        # ----------------------------------------------------------

        matches.to_csv(
            TEMP_FILE,
            sep="\t",
            index=False,
            mode="w" if first_write else "a",
            header=first_write
        )

        first_write = False

        total_matches += len(matches)

    total_processed += len(chunk)

    if chunk_no % 10 == 0:

        print(
            f"Processed : {total_processed:,} | "
            f"Matches : {total_matches:,}"
        )

    # --------------------------------------------------------------
    # RAM cleanup
    # --------------------------------------------------------------

    del X
    del probabilities
    del matches
    del chunk

    gc.collect()

print("=" * 70)

print("\nPrediction complete ✅")

print(f"Total candidates processed : {total_processed:,}")
print(f"High-confidence matches    : {total_matches:,}")

# ----------------------------------------------------------------------
# CHECK WHETHER MATCHES EXIST
# ----------------------------------------------------------------------

if total_matches == 0:

    raise ValueError(
        "No matches found. "
        "Try lowering MATCH_THRESHOLD from 0.80 to 0.70."
    )

# ----------------------------------------------------------------------
# CREATE FINAL SUBMISSION
# ----------------------------------------------------------------------

print("\nCreating final submission...")

# Read only the temporary matched records.
matches_df = pd.read_csv(
    TEMP_FILE,
    sep="\t",
    dtype={
        "source1_id": str,
        "target_id": str,
        "target_source": str
    }
)

print(f"Matched rows loaded : {len(matches_df):,}")

# ----------------------------------------------------------------------
# REMOVE DUPLICATE MATCHES
# ----------------------------------------------------------------------

matches_df = matches_df.drop_duplicates(
    subset=["source1_id", "target_id"]
)

print(
    f"After duplicate removal : "
    f"{len(matches_df):,}"
)

# ----------------------------------------------------------------------
# SORT BY PROBABILITY
# ----------------------------------------------------------------------

matches_df = matches_df.sort_values(
    ["source1_id", "match_probability"],
    ascending=[True, False]
)

# ----------------------------------------------------------------------
# GROUP TARGET IDs
# ----------------------------------------------------------------------

final_submission = (
    matches_df
    .groupby("source1_id", sort=False)["target_id"]
    .apply(lambda x: ",".join(dict.fromkeys(x.astype(str))))
    .reset_index()
)

# Rename exactly as required
final_submission.columns = [
    "source1_id",
    "matched_entity_ids"
]

# ----------------------------------------------------------------------
# SAVE FINAL CSV
# ----------------------------------------------------------------------

final_submission.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\n" + "=" * 70)
print("FINAL SUBMISSION CREATED ✅")
print("=" * 70)

print(f"Output file : {OUTPUT_FILE}")
print(f"Source-1 records with matches : {len(final_submission):,}")

# ----------------------------------------------------------------------
# DISPLAY SAMPLE
# ----------------------------------------------------------------------

print("\nSample final output:")
print("-" * 70)

print(
    final_submission.head(10).to_string(index=False)
)

print("\n" + "=" * 70)
print("CELL 5 COMPLETE ✅")
print("=" * 70)

CELL 5 - RAM-SAFE FINAL PREDICTION
Feature file found ✅
Model file found ✅
Feature file size : 1.97 GB

Loading trained model...
Model loaded ✅

Chunk size       : 100,000
Match threshold  : 0.8

Starting chunk-wise prediction...
Processed : 1,000,000 | Matches : 540,785
Processed : 2,000,000 | Matches : 1,082,595
Processed : 3,000,000 | Matches : 1,620,950
Processed : 4,000,000 | Matches : 2,164,627
Processed : 5,000,000 | Matches : 2,701,903
Processed : 6,000,000 | Matches : 3,244,372
Processed : 7,000,000 | Matches : 3,780,483
Processed : 8,000,000 | Matches : 4,322,181
Processed : 9,000,000 | Matches : 4,863,228
Processed : 10,000,000 | Matches : 5,404,733
Processed : 11,000,000 | Matches : 5,937,972
Processed : 12,000,000 | Matches : 6,478,243
Processed : 13,000,000 | Matches : 7,021,699
Processed : 14,000,000 | Matches : 7,566,626
Processed : 15,000,000 | Matches : 8,108,640
Processed : 16,000,000 | Matches : 8,652,770
Processed : 17,000,000 | Matches : 9,189,242
Processed : 18,0

In [8]:
# ======================================================================
# CELL 6 - FINAL MATCH QUALITY FILTERING
# ======================================================================

import os
import gc
import pandas as pd
import numpy as np

print("=" * 70)
print("CELL 6 - FINAL MATCH QUALITY FILTERING")
print("=" * 70)

FEATURE_FILE = "/content/drive/MyDrive/student_resource/blocking/ml_features_filtered.tsv"

FINAL_PREDICTION_FILE = "/content/drive/MyDrive/student_resource/blocking/final_submission.csv"

OUTPUT_FILE = "/content/drive/MyDrive/student_resource/blocking/final_submission_filtered.csv"

# ----------------------------------------------------------------------
# SETTINGS
# ----------------------------------------------------------------------

CHUNK_SIZE = 100_000

# Minimum ML probability
MIN_PROBABILITY = 0.80

# Quality score thresholds
HIGH_NAME = 0.90
HIGH_ADDRESS = 0.80

print("Feature file found:", os.path.exists(FEATURE_FILE))
print("Prediction file found:", os.path.exists(FINAL_PREDICTION_FILE))

# ----------------------------------------------------------------------
# STEP 1 - READ FINAL PREDICTION IDs
# ----------------------------------------------------------------------

print("\nReading predicted matches...")

predicted = pd.read_csv(
    FINAL_PREDICTION_FILE,
    dtype=str
)

print(f"Source-1 records in prediction : {len(predicted):,}")

# Convert comma-separated IDs into individual pairs
predicted["target_id"] = predicted["matched_entity_ids"].fillna("").str.split(",")

predicted = predicted.explode("target_id")

predicted["target_id"] = predicted["target_id"].astype(str).str.strip()

predicted = predicted[
    (predicted["target_id"] != "") &
    (predicted["target_id"] != "nan")
]

predicted = predicted[
    ["source1_id", "target_id"]
].drop_duplicates()

print(f"Unique predicted pairs : {len(predicted):,}")

# ----------------------------------------------------------------------
# STEP 2 - PROCESS ORIGINAL FEATURE FILE
# ----------------------------------------------------------------------

print("\nFiltering using original matching features...")
print("=" * 70)

USE_COLUMNS = [
    "source1_id",
    "target_id",
    "target_source",
    "name_similarity",
    "address_similarity",
    "country_match",
    "exact_name",
    "name_length_diff",
    "address_length_diff"
]

reader = pd.read_csv(
    FEATURE_FILE,
    sep="\t",
    usecols=USE_COLUMNS,
    chunksize=CHUNK_SIZE,
    low_memory=True
)

filtered_parts = []

processed = 0
kept = 0

for chunk_no, chunk in enumerate(reader, start=1):

    # --------------------------------------------------------------
    # Keep only pairs predicted by Cell 5
    # --------------------------------------------------------------

    chunk = chunk.merge(
        predicted,
        on=["source1_id", "target_id"],
        how="inner"
    )

    if len(chunk) == 0:
        continue

    # --------------------------------------------------------------
    # Convert numeric features
    # --------------------------------------------------------------

    numeric_cols = [
        "name_similarity",
        "address_similarity",
        "country_match",
        "exact_name",
        "name_length_diff",
        "address_length_diff"
    ]

    for col in numeric_cols:
        chunk[col] = pd.to_numeric(
            chunk[col],
            errors="coerce"
        ).fillna(0)

    # --------------------------------------------------------------
    # QUALITY RULES
    # --------------------------------------------------------------

    strong_name = (
        (chunk["name_similarity"] >= HIGH_NAME) |
        (chunk["exact_name"] == 1)
    )

    strong_address = (
        chunk["address_similarity"] >= HIGH_ADDRESS
    )

    country_ok = (
        chunk["country_match"] == 1
    )

    # --------------------------------------------------------------
    # FINAL QUALITY CONDITION
    #
    # Keep:
    #   Strong name + strong address
    # OR
    #   Exact name + reasonable address
    # OR
    #   Very strong name + country match
    # --------------------------------------------------------------

    quality_condition = (
        (strong_name & strong_address)
        |
        (
            (chunk["exact_name"] == 1) &
            (chunk["address_similarity"] >= 0.60)
        )
        |
        (
            (chunk["name_similarity"] >= 0.95) &
            country_ok
        )
    )

    good = chunk[quality_condition].copy()

    if len(good) > 0:

        filtered_parts.append(
            good[
                [
                    "source1_id",
                    "target_id",
                    "target_source",
                    "name_similarity",
                    "address_similarity",
                    "country_match",
                    "exact_name"
                ]
            ]
        )

        kept += len(good)

    processed += CHUNK_SIZE

    if chunk_no % 10 == 0:
        print(
            f"Processed : {processed:,} | "
            f"Quality matches kept : {kept:,}"
        )

    del chunk
    del good
    gc.collect()

# ----------------------------------------------------------------------
# STEP 3 - COMBINE FILTERED RESULTS
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("QUALITY FILTERING COMPLETE")
print("=" * 70)

if not filtered_parts:
    raise ValueError(
        "No high-quality matches survived the filtering rules."
    )

filtered = pd.concat(
    filtered_parts,
    ignore_index=True
)

del filtered_parts
gc.collect()

print(f"Filtered pairs : {len(filtered):,}")

# ----------------------------------------------------------------------
# STEP 4 - REMOVE DUPLICATES
# ----------------------------------------------------------------------

filtered = filtered.drop_duplicates(
    subset=["source1_id", "target_id"]
)

print(f"After duplicate removal : {len(filtered):,}")

# ----------------------------------------------------------------------
# STEP 5 - CREATE QUALITY SCORE
# ----------------------------------------------------------------------

filtered["quality_score"] = (
    filtered["name_similarity"] * 0.55
    +
    filtered["address_similarity"] * 0.35
    +
    filtered["country_match"] * 0.05
    +
    filtered["exact_name"] * 0.05
)

# ----------------------------------------------------------------------
# STEP 6 - SORT BEST MATCHES FIRST
# ----------------------------------------------------------------------

filtered = filtered.sort_values(
    ["source1_id", "quality_score"],
    ascending=[True, False]
)

# ----------------------------------------------------------------------
# STEP 7 - FINAL FORMAT
# ----------------------------------------------------------------------

final_submission = (
    filtered
    .groupby("source1_id", sort=False)["target_id"]
    .apply(
        lambda x: ",".join(
            dict.fromkeys(x.astype(str))
        )
    )
    .reset_index()
)

final_submission.columns = [
    "source1_id",
    "matched_entity_ids"
]

# ----------------------------------------------------------------------
# STEP 8 - SAVE
# ----------------------------------------------------------------------

final_submission.to_csv(
    OUTPUT_FILE,
    index=False
)

# ----------------------------------------------------------------------
# RESULTS
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL FILTERED SUBMISSION CREATED ✅")
print("=" * 70)

print(f"Output file : {OUTPUT_FILE}")
print(
    f"Source-1 records with matches : "
    f"{len(final_submission):,}"
)

print("\nSample:")
print("-" * 70)

print(
    final_submission.head(10).to_string(index=False)
)

print("\n" + "=" * 70)
print("CELL 6 COMPLETE ✅")
print("=" * 70)

CELL 6 - FINAL MATCH QUALITY FILTERING
Feature file found: True
Prediction file found: True

Reading predicted matches...
Source-1 records in prediction : 1,680,277
Unique predicted pairs : 14,854,414

Filtering using original matching features...
Processed : 1,000,000 | Quality matches kept : 300,387
Processed : 2,000,000 | Quality matches kept : 602,419
Processed : 3,000,000 | Quality matches kept : 902,991
Processed : 4,000,000 | Quality matches kept : 1,205,590
Processed : 5,000,000 | Quality matches kept : 1,504,186
Processed : 6,000,000 | Quality matches kept : 1,806,780
Processed : 7,000,000 | Quality matches kept : 2,105,725
Processed : 8,000,000 | Quality matches kept : 2,407,623
Processed : 9,000,000 | Quality matches kept : 2,709,728
Processed : 10,000,000 | Quality matches kept : 3,012,364
Processed : 11,000,000 | Quality matches kept : 3,308,354
Processed : 12,000,000 | Quality matches kept : 3,610,034
Processed : 13,000,000 | Quality matches kept : 3,912,920
Processed : 1

In [9]:
# ======================================================================
# CELL 7 - FINAL SUBMISSION VALIDATION
# ======================================================================

import os
import pandas as pd
import gc

print("=" * 70)
print("CELL 7 - FINAL SUBMISSION VALIDATION")
print("=" * 70)

FINAL_FILE = "/content/drive/MyDrive/student_resource/blocking/final_submission_filtered.csv"

# ----------------------------------------------------------------------
# STEP 1 - FILE CHECK
# ----------------------------------------------------------------------

if not os.path.exists(FINAL_FILE):
    raise FileNotFoundError(
        f"Final file not found:\n{FINAL_FILE}"
    )

file_size_mb = os.path.getsize(FINAL_FILE) / (1024 * 1024)

print(f"Final file found ✅")
print(f"File size : {file_size_mb:.2f} MB")

# ----------------------------------------------------------------------
# STEP 2 - LOAD FINAL FILE
# ----------------------------------------------------------------------

df = pd.read_csv(
    FINAL_FILE,
    dtype=str
)

print("\nFile loaded successfully ✅")

print(f"Rows : {len(df):,}")
print(f"Columns : {list(df.columns)}")

# ----------------------------------------------------------------------
# STEP 3 - COLUMN VALIDATION
# ----------------------------------------------------------------------

EXPECTED_COLUMNS = [
    "source1_id",
    "matched_entity_ids"
]

if list(df.columns) == EXPECTED_COLUMNS:
    print("\nColumn format : PASS ✅")
else:
    print("\nColumn format : FAIL ❌")
    print("Expected :", EXPECTED_COLUMNS)
    print("Found    :", list(df.columns))

# ----------------------------------------------------------------------
# STEP 4 - SOURCE ID VALIDATION
# ----------------------------------------------------------------------

invalid_source = df[
    ~df["source1_id"].str.startswith("S1-", na=False)
]

print("\nSource-1 ID validation")
print("-" * 70)
print(f"Invalid source1 IDs : {len(invalid_source):,}")

if len(invalid_source) == 0:
    print("Source-1 IDs : PASS ✅")
else:
    print("Source-1 IDs : CHECK ❌")

# ----------------------------------------------------------------------
# STEP 5 - EMPTY MATCH VALIDATION
# ----------------------------------------------------------------------

empty_matches = df[
    df["matched_entity_ids"].isna()
    |
    (df["matched_entity_ids"].str.strip() == "")
]

print("\nEmpty match validation")
print("-" * 70)
print(f"Empty match rows : {len(empty_matches):,}")

if len(empty_matches) == 0:
    print("Empty matches : PASS ✅")
else:
    print("Empty matches : CHECK ⚠️")

# ----------------------------------------------------------------------
# STEP 6 - DUPLICATE SOURCE VALIDATION
# ----------------------------------------------------------------------

duplicate_source_count = df["source1_id"].duplicated().sum()

print("\nSource-1 duplicate validation")
print("-" * 70)
print(f"Duplicate source1 rows : {duplicate_source_count:,}")

if duplicate_source_count == 0:
    print("Source-1 uniqueness : PASS ✅")
else:
    print("Source-1 uniqueness : FAIL ❌")

# ----------------------------------------------------------------------
# STEP 7 - TARGET ID VALIDATION
# ----------------------------------------------------------------------

print("\nChecking target IDs...")
print("-" * 70)

invalid_target_count = 0
total_target_count = 0

for value in df["matched_entity_ids"]:

    if pd.isna(value):
        continue

    ids = str(value).split(",")

    for target_id in ids:

        target_id = target_id.strip()

        if target_id == "":
            continue

        total_target_count += 1

        if not (
            target_id.startswith("S2-")
            or target_id.startswith("S3-")
        ):
            invalid_target_count += 1

print(f"Total target IDs : {total_target_count:,}")
print(f"Invalid target IDs : {invalid_target_count:,}")

if invalid_target_count == 0:
    print("Target ID format : PASS ✅")
else:
    print("Target ID format : FAIL ❌")

# ----------------------------------------------------------------------
# STEP 8 - DUPLICATE TARGET IDS INSIDE EACH SOURCE
# ----------------------------------------------------------------------

duplicate_inside_count = 0

for value in df["matched_entity_ids"]:

    if pd.isna(value):
        continue

    ids = [
        x.strip()
        for x in str(value).split(",")
        if x.strip()
    ]

    if len(ids) != len(set(ids)):
        duplicate_inside_count += 1

print("\nDuplicate target validation")
print("-" * 70)
print(
    f"Rows containing duplicate target IDs : "
    f"{duplicate_inside_count:,}"
)

if duplicate_inside_count == 0:
    print("Internal duplicates : PASS ✅")
else:
    print("Internal duplicates : CHECK ❌")

# ----------------------------------------------------------------------
# STEP 9 - MATCH COUNT DISTRIBUTION
# ----------------------------------------------------------------------

df["match_count"] = (
    df["matched_entity_ids"]
    .fillna("")
    .apply(
        lambda x: len(
            [i for i in str(x).split(",") if i.strip()]
        )
    )
)

print("\nMatch count statistics")
print("-" * 70)

print(f"Minimum matches : {df['match_count'].min():,}")
print(f"Maximum matches : {df['match_count'].max():,}")
print(f"Average matches : {df['match_count'].mean():.2f}")
print(f"Median matches  : {df['match_count'].median():.2f}")

# ----------------------------------------------------------------------
# STEP 10 - HIGH MATCH COUNT RECORDS
# ----------------------------------------------------------------------

print("\nTop 10 source records with highest number of matches")
print("-" * 70)

top_matches = (
    df[
        ["source1_id", "match_count"]
    ]
    .sort_values(
        "match_count",
        ascending=False
    )
    .head(10)
)

print(
    top_matches.to_string(index=False)
)

# ----------------------------------------------------------------------
# STEP 11 - SAMPLE OUTPUT
# ----------------------------------------------------------------------

print("\nSample final output")
print("-" * 70)

print(
    df[
        ["source1_id", "matched_entity_ids"]
    ]
    .head(10)
    .to_string(index=False)
)

# ----------------------------------------------------------------------
# STEP 12 - FINAL STATUS
# ----------------------------------------------------------------------

all_pass = (
    list(df.columns) == EXPECTED_COLUMNS
    and len(invalid_source) == 0
    and len(empty_matches) == 0
    and duplicate_source_count == 0
    and invalid_target_count == 0
    and duplicate_inside_count == 0
)

print("\n" + "=" * 70)

if all_pass:
    print("FINAL VALIDATION : PASS ✅")
    print("File structure is submission-ready ✅")
else:
    print("FINAL VALIDATION : CHECK REQUIRED ⚠️")

print("=" * 70)

print(f"\nFinal file:")
print(FINAL_FILE)

del df
gc.collect()

print("\n" + "=" * 70)
print("CELL 7 COMPLETE ✅")
print("=" * 70)

CELL 7 - FINAL SUBMISSION VALIDATION
Final file found ✅
File size : 119.88 MB

File loaded successfully ✅
Rows : 1,355,326
Columns : ['source1_id', 'matched_entity_ids']

Column format : PASS ✅

Source-1 ID validation
----------------------------------------------------------------------
Invalid source1 IDs : 0
Source-1 IDs : PASS ✅

Empty match validation
----------------------------------------------------------------------
Empty match rows : 0
Empty matches : PASS ✅

Source-1 duplicate validation
----------------------------------------------------------------------
Duplicate source1 rows : 0
Source-1 uniqueness : PASS ✅

Checking target IDs...
----------------------------------------------------------------------
Total target IDs : 8,274,463
Invalid target IDs : 0
Target ID format : PASS ✅

Duplicate target validation
----------------------------------------------------------------------
Rows containing duplicate target IDs : 0
Internal duplicates : PASS ✅

Match count statistics
-

In [10]:
# ======================================================================
# CELL 8 - MATCH DISTRIBUTION ANALYSIS
# ======================================================================

import pandas as pd
import os

print("=" * 70)
print("CELL 8 - FINAL MATCH DISTRIBUTION ANALYSIS")
print("=" * 70)

FINAL_FILE = "/content/drive/MyDrive/student_resource/blocking/final_submission_filtered.csv"

# Load only for analysis
df = pd.read_csv(FINAL_FILE, dtype=str)

# Count matched entities
df["match_count"] = (
    df["matched_entity_ids"]
    .fillna("")
    .apply(lambda x: len([i for i in str(x).split(",") if i.strip()]))
)

print(f"\nTotal source-1 records : {len(df):,}")
print(f"Total matched pairs     : {df['match_count'].sum():,}")

print("\n" + "=" * 70)
print("MATCH COUNT DISTRIBUTION")
print("=" * 70)

# Distribution
distribution = pd.cut(
    df["match_count"],
    bins=[0, 1, 5, 10, 20, 50, 1000000],
    labels=[
        "1 match",
        "2-5 matches",
        "6-10 matches",
        "11-20 matches",
        "21-50 matches",
        "51+ matches"
    ],
    include_lowest=True
)

dist = distribution.value_counts().sort_index()

for category, count in dist.items():
    percentage = count / len(df) * 100
    print(
        f"{str(category):15} : "
        f"{count:10,} records "
        f"({percentage:6.2f}%)"
    )

# ----------------------------------------------------------------------
# TOP HIGH-MATCH RECORDS
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("TOP 20 HIGHEST-MATCH SOURCE RECORDS")
print("=" * 70)

top20 = (
    df[["source1_id", "match_count"]]
    .sort_values("match_count", ascending=False)
    .head(20)
)

print(top20.to_string(index=False))

# ----------------------------------------------------------------------
# IMPORTANT: CHECK SOURCE IDs ARE STILL ORIGINAL FORMAT
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("SOURCE ID SAFETY CHECK")
print("=" * 70)

invalid_ids = df[
    ~df["source1_id"].str.startswith("S1-", na=False)
]

duplicate_ids = df["source1_id"].duplicated().sum()

print(f"Invalid Source-1 IDs : {len(invalid_ids):,}")
print(f"Duplicate Source-1 IDs : {duplicate_ids:,}")

if len(invalid_ids) == 0 and duplicate_ids == 0:
    print("Source-1 IDs preserved correctly ✅")
else:
    print("SOURCE ID ISSUE FOUND ❌")

# ----------------------------------------------------------------------
# FINAL FORMAT CHECK
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL FORMAT CHECK")
print("=" * 70)

print("Columns:")
print(list(df[["source1_id", "matched_entity_ids"]].columns))

if list(df[["source1_id", "matched_entity_ids"]].columns) == [
    "source1_id",
    "matched_entity_ids"
]:
    print("Final format unchanged ✅")
else:
    print("Final format changed ❌")

# ----------------------------------------------------------------------
# DO NOT SAVE / MODIFY FINAL FILE
# ----------------------------------------------------------------------

del df

print("\n" + "=" * 70)
print("CELL 8 COMPLETE ✅")
print("Original final_submission_filtered.csv was NOT modified.")
print("=" * 70)

CELL 8 - FINAL MATCH DISTRIBUTION ANALYSIS

Total source-1 records : 1,355,326
Total matched pairs     : 8,274,463

MATCH COUNT DISTRIBUTION
1 match         :    558,628 records ( 41.22%)
2-5 matches     :    458,651 records ( 33.84%)
6-10 matches    :    159,952 records ( 11.80%)
11-20 matches   :     71,936 records (  5.31%)
21-50 matches   :     85,367 records (  6.30%)
51+ matches     :     20,792 records (  1.53%)

TOP 20 HIGHEST-MATCH SOURCE RECORDS
  source1_id  match_count
S1-264491120           94
S1-140114822           94
S1-168773850           90
S1-490727471           82
S1-664885317           82
S1-980572399           82
S1-409214726           82
S1-660005617           81
S1-648526173           81
S1-259072036           81
S1-463895915           81
S1-894302642           81
S1-755934967           81
S1-811158509           81
S1-552189114           81
S1-117109689           81
S1-829555582           81
 S1-33217418           81
S1-208888674           81
 S1-23821853        

In [11]:
# ======================================================================
# CELL 9 - FINAL BACKUP + INTEGRITY CHECK
# ======================================================================

import os
import shutil
import pandas as pd

print("=" * 70)
print("CELL 9 - FINAL BACKUP + INTEGRITY CHECK")
print("=" * 70)

FINAL_FILE = "/content/drive/MyDrive/student_resource/blocking/final_submission_filtered.csv"

BACKUP_FILE = "/content/drive/MyDrive/student_resource/blocking/final_submission_filtered_backup.csv"

# ----------------------------------------------------------------------
# STEP 1 - CHECK ORIGINAL FILE
# ----------------------------------------------------------------------

if not os.path.exists(FINAL_FILE):
    raise FileNotFoundError(
        f"Final submission file not found:\n{FINAL_FILE}"
    )

print("Original final file found ✅")

original_size = os.path.getsize(FINAL_FILE)

print(f"Original file size : {original_size / (1024 * 1024):.2f} MB")

# ----------------------------------------------------------------------
# STEP 2 - LOAD AND VERIFY
# ----------------------------------------------------------------------

df = pd.read_csv(
    FINAL_FILE,
    dtype=str
)

print("\nFinal file loaded ✅")

print(f"Rows : {len(df):,}")
print(f"Columns : {list(df.columns)}")

# ----------------------------------------------------------------------
# STEP 3 - FORMAT CHECK
# ----------------------------------------------------------------------

EXPECTED_COLUMNS = [
    "source1_id",
    "matched_entity_ids"
]

if list(df.columns) != EXPECTED_COLUMNS:
    raise ValueError(
        f"Final format changed!\n"
        f"Expected: {EXPECTED_COLUMNS}\n"
        f"Found: {list(df.columns)}"
    )

print("Final format : PASS ✅")

# ----------------------------------------------------------------------
# STEP 4 - SOURCE ID CHECK
# ----------------------------------------------------------------------

invalid_source_ids = df[
    ~df["source1_id"].str.startswith("S1-", na=False)
]

duplicate_source_ids = df["source1_id"].duplicated().sum()

print("\nSource-1 validation")
print("-" * 70)
print(f"Invalid Source-1 IDs : {len(invalid_source_ids):,}")
print(f"Duplicate Source-1 IDs : {duplicate_source_ids:,}")

if len(invalid_source_ids) != 0:
    raise ValueError("Invalid Source-1 IDs found!")

if duplicate_source_ids != 0:
    raise ValueError("Duplicate Source-1 IDs found!")

print("Source-1 IDs preserved 🔒")

# ----------------------------------------------------------------------
# STEP 5 - MATCH VALIDATION
# ----------------------------------------------------------------------

empty_matches = df[
    df["matched_entity_ids"].isna()
    |
    (df["matched_entity_ids"].str.strip() == "")
]

invalid_targets = 0
total_targets = 0

for value in df["matched_entity_ids"]:

    if pd.isna(value):
        continue

    ids = [
        x.strip()
        for x in str(value).split(",")
        if x.strip()
    ]

    for target_id in ids:

        total_targets += 1

        if not (
            target_id.startswith("S2-")
            or target_id.startswith("S3-")
        ):
            invalid_targets += 1

print("\nTarget validation")
print("-" * 70)
print(f"Total target IDs : {total_targets:,}")
print(f"Invalid target IDs : {invalid_targets:,}")
print(f"Empty match rows : {len(empty_matches):,}")

if invalid_targets != 0:
    raise ValueError("Invalid target IDs found!")

if len(empty_matches) != 0:
    raise ValueError("Empty matches found!")

print("Target IDs valid ✅")

# ----------------------------------------------------------------------
# STEP 6 - CREATE BACKUP
# ----------------------------------------------------------------------

print("\nCreating backup...")

shutil.copy2(
    FINAL_FILE,
    BACKUP_FILE
)

print("Backup created ✅")

# ----------------------------------------------------------------------
# STEP 7 - BACKUP SIZE CHECK
# ----------------------------------------------------------------------

backup_size = os.path.getsize(BACKUP_FILE)

print(f"Original size : {original_size:,} bytes")
print(f"Backup size   : {backup_size:,} bytes")

if original_size != backup_size:
    raise ValueError(
        "Backup size does not match original file!"
    )

print("Backup integrity : PASS ✅")

# ----------------------------------------------------------------------
# STEP 8 - FINAL STATUS
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL SUBMISSION STATUS")
print("=" * 70)

print("Original final file       : SAFE ✅")
print("Source-1 IDs              : PRESERVED 🔒")
print("Final format              : PRESERVED 🔒")
print("Target IDs                : VALID ✅")
print("Empty matches             : NONE ✅")
print("Backup                    : CREATED ✅")
print("=" * 70)

print("\nFINAL FILE:")
print(FINAL_FILE)

print("\nBACKUP FILE:")
print(BACKUP_FILE)

print("\n" + "=" * 70)
print("CELL 9 COMPLETE ✅")
print("=" * 70)

del df

CELL 9 - FINAL BACKUP + INTEGRITY CHECK
Original final file found ✅
Original file size : 119.88 MB

Final file loaded ✅
Rows : 1,355,326
Columns : ['source1_id', 'matched_entity_ids']
Final format : PASS ✅

Source-1 validation
----------------------------------------------------------------------
Invalid Source-1 IDs : 0
Duplicate Source-1 IDs : 0
Source-1 IDs preserved 🔒

Target validation
----------------------------------------------------------------------
Total target IDs : 8,274,463
Invalid target IDs : 0
Empty match rows : 0
Target IDs valid ✅

Creating backup...
Backup created ✅
Original size : 125,707,619 bytes
Backup size   : 125,707,619 bytes
Backup integrity : PASS ✅

FINAL SUBMISSION STATUS
Original final file       : SAFE ✅
Source-1 IDs              : PRESERVED 🔒
Final format              : PRESERVED 🔒
Target IDs                : VALID ✅
Empty matches             : NONE ✅
Backup                    : CREATED ✅

FINAL FILE:
/content/drive/MyDrive/student_resource/blocking/f